In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. LOAD DATA
df = pd.read_csv('ai_financial.csv')

# 2. CLEAN HEADERS (Remove spaces/special chars)
df.columns = df.columns.str.strip()

# 3. POSITION-BASED MAPPING
# This assigns names based on the order in the CSV to avoid KeyError
# Change the numbers [0, 1, 2...] if your CSV columns are in a different order
cols = {
    'date': df.columns[0],      # e.g., 'Date'
    'company': df.columns[1],   # e.g., 'Company'
    'rd': df.columns[2],        # e.g., 'R&D_Spent'
    'revenue': df.columns[3],   # e.g., 'AI_Revenue'
    'growth': df.columns[4],    # e.g., 'AI_Revenue_Growth'
    'event': df.columns[5],     # e.g., 'Event'
    'impact': df.columns[6]     # e.g., 'Stock_Impact'
}

print("Mapping Successful:")
for key, val in cols.items():
    print(f"  {key} -> {val}")

# 4. DATA CONVERSION
numeric_fields = [cols['rd'], cols['revenue'], cols['growth'], cols['impact']]
for field in numeric_fields:
    df[field] = pd.to_numeric(df[field], errors='coerce').fillna(0)

# Convert Date and create Year
df[cols['date']] = pd.to_datetime(df[cols['date']], errors='coerce')
df['Year'] = df[cols['date']].dt.year


# 1) Amount spent on R & D per Company
print("\n--- 1) Total R&D Investment per Company ---")
print(df.groupby(cols['company'])[cols['rd']].sum())

# 2) Revenue Earned
print("\n--- 2) Total Revenue Earned ---")
print(df.groupby(cols['company'])[cols['revenue']].sum())

# 3) Date-wise Impact on Stock (Line Graph)
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x=cols['date'], y=cols['impact'], hue=cols['company'])
plt.title('AI Market: Daily Stock Impact (2015-2024)')
plt.grid(True)
plt.show()

# 4) Maximum Stock Impact Events
print("\n--- 4) Top 5 High-Impact Events ---")
max_events = df.sort_values(by=cols['impact'], ascending=False).head(5)
print(max_events[[cols['date'], cols['company'], cols['event'], cols['impact']]])

# 6) Correlation Analysis
plt.figure(figsize=(10, 8))
# Correlation only between the actual numeric columns
correlation_data = df[numeric_fields].corr()
sns.heatmap(correlation_data, annot=True, cmap='RdYlGn', fmt=".2f")
plt.title('Financial Metrics Correlation Matrix')
plt.show()

# 7) Expenditure vs Revenue year-by-year
plt.figure(figsize=(12, 6))
yearly_comparison = df.groupby('Year')[[cols['rd'], cols['revenue']]].sum()
yearly_comparison.plot(kind='bar', figsize=(12, 6))
plt.title('Annual R&D Expenditure vs. Revenue')
plt.ylabel('Amount (in Billions)')
plt.show()

# 8) Event Impact Analysis
print("\n--- 8) Average Stock Impact by Event Type ---")
event_impact = df.groupby(cols['event'])[cols['impact']].mean().sort_values(ascending=False)
print(event_impact)

# 9) Pivot Table: Yearly Avg Stock Impact per Company
print("\n--- 9) Yearly Avg Stock Impact Index ---")
pivot = df.pivot_table(index='Year', columns=cols['company'], values=cols['impact'], aggfunc='mean')
print(pivot)